# 03 — Regression Discontinuity Design (RDD)
**Autores clave:** Thistlethwaite & Campbell (1960) · Hahn, Todd & Van der Klaauw (2001) · Imbens & Lemieux (2008) · Lee & Lemieux (2010) · Calonico, Cattaneo & Titiunik (2014)

## Intuición
Cerca del umbral de asignación $c$, las unidades con score $X_i$ son casi idénticas — su única diferencia es si recibieron tratamiento. Esto crea cuasi-aleatorización local.

## RDD Sharp (Hahn, Todd & Van der Klaauw, 2001)
$$D_i = \mathbb{1}[X_i \geq c]$$
$$\tau_{RD} = \lim_{x \downarrow c} E[Y_i|X_i=x] - \lim_{x \uparrow c} E[Y_i|X_i=x]$$

## RDD Fuzzy
El tratamiento no es determinístico en el umbral (ej. elegibilidad $\neq$ participación). El estimador es equivalente a un IV local:
$$\tau_{Fuzzy} = \frac{\lim_{x\downarrow c}E[Y|X=x] - \lim_{x\uparrow c}E[Y|X=x]}{\lim_{x\downarrow c}E[D|X=x] - \lim_{x\uparrow c}E[D|X=x]}$$

## Bandwidth
El bandwidth $h$ controla el sesgo-varianza: grande → más datos pero más sesgo; pequeño → menos sesgo pero más ruido. Imbens & Kalyanaraman (2012) y Calonico, Cattaneo & Titiunik (2014) proponen selección óptima.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# Contexto: efecto de recibir beca académica en rendimiento futuro
# Beca: puntaje >= 60 (umbral c = 60)
n = 2000
c = 60  # umbral (cutoff)
TRUE_EFFECT = 5.0  # efecto verdadero de la beca

# Score de calificación (running variable) — continua, centrada en c
score = np.random.uniform(30, 90, n)

# Sharp RD: tratamiento determinístico en el umbral
D = (score >= c).astype(float)

# Outcome: tendencia suave del score + salto en c
# Función cuadrática subyacente (diferente a cada lado)
y0 = 40 + 0.5*(score - c) - 0.003*(score - c)**2 + np.random.normal(0, 3, n)
y1 = y0 + TRUE_EFFECT
Y  = np.where(D == 1, y1, y0)

df = pd.DataFrame({'Y': Y, 'score': score, 'D': D, 'score_c': score - c})
print(f'Sharp RD — umbral c={c}, efecto verdadero τ={TRUE_EFFECT}')
print(f'Tratados: {D.sum():.0f} ({D.mean():.1%})')

## 1 — Gráfica canónica de RD

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: scatter con bins
n_bins = 30
bins   = np.linspace(df['score'].min(), df['score'].max(), n_bins + 1)
df['bin'] = pd.cut(df['score'], bins=bins)
bin_means = df.groupby('bin', observed=True).agg(
    Y_mean=('Y', 'mean'), score_mean=('score', 'mean'), D_mean=('D', 'mean')
).reset_index()

left  = bin_means[bin_means['score_mean'] <  c]
right = bin_means[bin_means['score_mean'] >= c]

axes[0].scatter(left['score_mean'],  left['Y_mean'],  s=40, color='#4361ee', zorder=5, label='Control')
axes[0].scatter(right['score_mean'], right['Y_mean'], s=40, color='#f72585', zorder=5, label='Tratado')

# Fit lineal en cada lado
for side_df, color in [(df[df['score'] < c], '#4361ee'), (df[df['score'] >= c], '#f72585')]:
    sl, ic, *_ = stats.linregress(side_df['score_c'], side_df['Y'])
    x_range = np.linspace(side_df['score'].min(), side_df['score'].max(), 100)
    axes[0].plot(x_range, ic + sl*(x_range - c), color=color, linewidth=2)

axes[0].axvline(c, color='black', linewidth=1.5, linestyle='--', label=f'Umbral c={c}')
axes[0].set_xlabel('Score (running variable)')
axes[0].set_ylabel('Outcome Y')
axes[0].set_title(f'Sharp RD — Thistlethwaite & Campbell (1960)\nSalto en c = {c}')
axes[0].legend(fontsize=9)

# Panel B: densidad del score (McCrary test visual)
axes[1].hist(df[df['D']==0]['score'], bins=30, color='#4361ee', alpha=0.6, label='Control', density=True)
axes[1].hist(df[df['D']==1]['score'], bins=30, color='#f72585', alpha=0.6, label='Tratado', density=True)
axes[1].axvline(c, color='black', linewidth=1.5, linestyle='--', label=f'c={c}')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Densidad')
axes[1].set_title('Densidad del score — McCrary (2008)\nSalto en c sugiere manipulación')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 2 — Estimación RD: OLS local con bandwidth

In [ ]:
def estimate_rd(df, h, poly_order=1):
    """Estima el efecto RD con regresión local de orden poly_order dentro de bandwidth h."""
    mask = np.abs(df['score_c']) <= h
    d_sub = df[mask].copy()
    if len(d_sub) < 10:
        return np.nan, np.nan, np.nan

    # Construir regresores: D, score_c, D*score_c (interacción para pendiente diferente a cada lado)
    X_cols = {'const': np.ones(len(d_sub)), 'D': d_sub['D'].values,
              'score_c': d_sub['score_c'].values, 'D_score_c': (d_sub['D'] * d_sub['score_c']).values}
    if poly_order == 2:
        X_cols['score_c2']   = d_sub['score_c'].values**2
        X_cols['D_score_c2'] = (d_sub['D'] * d_sub['score_c']**2).values

    X_mat = pd.DataFrame(X_cols)
    mod   = sm.OLS(d_sub['Y'], X_mat).fit()
    return mod.params['D'], mod.bse['D'], len(d_sub)

bandwidths = [5, 10, 15, 20, 25, 30]
rd_results = []
for h in bandwidths:
    coef1, se1, n1 = estimate_rd(df, h, poly_order=1)
    coef2, se2, n2 = estimate_rd(df, h, poly_order=2)
    rd_results.append({'h': h, 'n_obs': n1,
                       'coef_lin': coef1, 'se_lin': se1,
                       'coef_quad': coef2, 'se_quad': se2})

rd_df = pd.DataFrame(rd_results)
print(f'Estimación RD (τ verdadero = {TRUE_EFFECT})')
print(rd_df.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar(rd_df['h'], rd_df['coef_lin'], yerr=1.96*rd_df['se_lin'],
            fmt='o-', color='#4361ee', capsize=4, linewidth=2, markersize=7, label='Local lineal')
ax.errorbar(rd_df['h'] + 0.3, rd_df['coef_quad'], yerr=1.96*rd_df['se_quad'],
            fmt='s--', color='#f72585', capsize=4, linewidth=2, markersize=7, label='Local cuadrática')
ax.axhline(TRUE_EFFECT, color='black', linestyle=':', linewidth=2, label=f'τ verdadero = {TRUE_EFFECT}')
ax.set_xlabel('Bandwidth h')
ax.set_ylabel('Estimado RD (IC 95%)')
ax.set_title('Sensibilidad al bandwidth — Imbens & Lemieux (2008)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 3 — Tests de validación

Un RD válido requiere que las **características predeterminadas** (pre-tratamiento) no salten en el umbral. Si saltan, el supuesto de continuidad falla.

In [ ]:
# Agregar covariables pre-tratamiento
df['edad']    = np.random.randint(18, 35, n).astype(float)    # no debe saltar en c
df['genero']  = np.random.binomial(1, 0.5, n).astype(float)   # no debe saltar en c
df['ingreso'] = np.random.lognormal(3, 0.5, n)                # no debe saltar en c

# Introducir una covariable manipulada (test de falsificación fallido)
df['edad_manipulada'] = df['edad'] + 2.0 * df['D']  # sí salta — inválido

h = 15
covariates = {
    'Edad (sin manipulación)':    'edad',
    'Género':                     'genero',
    'Ingreso familiar':           'ingreso',
    'Edad (manipulada — falla)':  'edad_manipulada',
}

print('Tests de Falsificación — Imbens & Lemieux (2008)')
print('H₀: covariable no salta en el umbral (p > 0.05 = válido)')
print('─' * 65)
for label, col in covariates.items():
    coef, se, n_h = estimate_rd(df.rename(columns={col: 'Y_temp'}).assign(Y=df[col]), h)
    t    = coef / se if se > 0 else np.nan
    p    = 2 * (1 - stats.norm.cdf(abs(t))) if not np.isnan(t) else np.nan
    flag = '✅' if p > 0.05 else '❌ FALLA'
    print(f'  {label:<35} coef={coef:>7.3f}  p={p:.4f}  {flag}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col, title in [
    (axes[0], 'edad',           'Edad — sin salto (✅ válido)'),
    (axes[1], 'edad_manipulada','Edad manipulada — salto (❌ falla)'),
]:
    mask = np.abs(df['score_c']) <= h
    d_sub = df[mask]
    for side, color, lbl in [(d_sub[d_sub['D']==0], '#4361ee', 'Control'),
                              (d_sub[d_sub['D']==1], '#f72585', 'Tratado')]:
        ax.scatter(side['score'], side[col], s=5, alpha=0.3, color=color)
        sl, ic, *_ = stats.linregress(side['score_c'], side[col])
        xr = np.linspace(side['score'].min(), side['score'].max(), 100)
        ax.plot(xr, ic + sl*(xr - c), color=color, linewidth=2, label=lbl)
    ax.axvline(c, color='black', linestyle='--', linewidth=1.5)
    ax.set_title(title); ax.set_xlabel('Score'); ax.set_ylabel(col)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Resumen

| Paso | Qué verificar | Herramienta |
|---|---|---|
| 1. Diseño | ¿El umbral determina el tratamiento? | Gráfica D vs score |
| 2. Visualización | Salto en Y en el umbral | Binned scatter + fit por lado |
| 3. Densidad | Sin manipulación del score | Histograma, McCrary (2008) |
| 4. Covariables | No saltan en el umbral | Falsification tests |
| 5. Estimación | Efecto causal local | Local lineal / cuadrática |
| 6. Robustez | Estabilidad al bandwidth | Gráfica coef vs h |

**Referencias:**
- Thistlethwaite, D.L. & Campbell, D.T. (1960). Regression-discontinuity analysis. *Journal of Educational Psychology*, 51(6).
- Imbens, G.W. & Lemieux, T. (2008). Regression discontinuity designs. *Journal of Econometrics*, 142(2).
- Lee, D.S. & Lemieux, T. (2010). Regression discontinuity designs in economics. *Journal of Economic Literature*, 48(2).
- Calonico, S., Cattaneo, M.D. & Titiunik, R. (2014). Robust nonparametric confidence intervals for RDD. *Econometrica*, 82(6).
- McCrary, J. (2008). Manipulation of the running variable in the regression discontinuity design. *Journal of Econometrics*, 142(2).

**Siguiente:** `04_potential_outcomes_dag.ipynb`